第五节，评估

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_classic.chains import RetrievalQA
from langchain_classic.chat_models import ChatOpenAI
from langchain_classic.document_loaders import CSVLoader
from langchain_classic.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown

In [3]:
file = "../data/data.csv"
loader = CSVLoader(file_path=file)

In [4]:
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_community.embeddings import DashScopeEmbeddings

In [5]:
embeddings = DashScopeEmbeddings(
    model="text-embedding-v1",  # 通义文本向量常用模型名
    dashscope_api_key=None,     # 不填则默认走环境变量 DASHSCOPE_API_KEY
)

In [6]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

In [7]:
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai",
    model="LongCat-Flash-Chat",
    max_tokens=1000,
    temperature=0.9
)

C:\Users\35186\AppData\Local\Temp\ipykernel_31768\2226652031.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [8]:
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs={
        "document_separator": "\n\n"
        }
)


In [9]:
data = loader.load()
data

[Document(metadata={'source': '../data/data.csv', 'row': 0}, page_content='name: T-shirt\ndescription: A comfortable T-shirt\nprice: 100\nsunscreen: Yes'),
 Document(metadata={'source': '../data/data.csv', 'row': 1}, page_content='name: Jeans\ndescription: A pair of jeans\nprice: 200\nsunscreen: No'),
 Document(metadata={'source': '../data/data.csv', 'row': 2}, page_content='name: Shoes\ndescription: A pair of shoes\nprice: 300\nsunscreen: No'),
 Document(metadata={'source': '../data/data.csv', 'row': 3}, page_content='name: Sunglasses\ndescription: A pair of sunglasses\nprice: 400\nsunscreen: Yes'),
 Document(metadata={'source': '../data/data.csv', 'row': 4}, page_content='name: Sunscreen\ndescription: A bottle of sunscreen\nprice: 500\nsunscreen: Yes'),
 Document(metadata={'source': '../data/data.csv', 'row': 5}, page_content='name: Apple\ndescription: A fresh apple\nprice: 600\nsunscreen: No'),
 Document(metadata={'source': '../data/data.csv', 'row': 6}, page_content='name: Banana\n

In [10]:
examples = [
    {
        "query": "列出所有有防晒功能的T恤，用markdown格式输出，一张表格中，含有总结",
        "answer": "T-shirt"
    }
]

In [11]:
from langchain_classic.evaluation.qa import QAGenerateChain

In [12]:
example_gen_chain = QAGenerateChain.from_llm(
    llm
)

In [13]:
new_examples = example_gen_chain.apply_and_parse([{"doc": doc} for doc in data[:5]])

C:\Users\35186\AppData\Local\Temp\ipykernel_31768\1324720881.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse([{"doc": doc} for doc in data[:5]])
d:\Python_Project\Notes_Lang\.venv\lib\site-packages\langchain_community\chat_models\openai.py:174: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [14]:
new_examples[0]["qa_pairs"]

{'query': 'According to the document, what is the price of the T-shirt, and does it have sunscreen protection?',
 'answer': 'The price of the T-shirt is 100, and it has sunscreen protection, as indicated by the value "Yes" under the sunscreen attribute.'}

In [15]:
for new_example in new_examples:
    examples.append(new_example["qa_pairs"])

In [16]:
qa.run(examples[0]["query"])

C:\Users\35186\AppData\Local\Temp\ipykernel_31768\1223946598.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  qa.run(examples[0]["query"])




> Entering new RetrievalQA chain...

> Finished chain.


'```markdown\n| 名称     | 描述               | 价格 | 防晒功能 |\n|----------|--------------------|------|----------|\n| T-shirt  | 一件舒适的T恤       | 100  | 是       |\n\n**总结**：共有 1 件T恤具备防晒功能。\n```'

In [17]:
from langchain_classic.globals import set_debug
set_debug(True)

In [18]:
qa.run(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "列出所有有防晒功能的T恤，用markdown格式输出，一张表格中，含有总结"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "列出所有有防晒功能的T恤，用markdown格式输出，一张表格中，含有总结",
  "context": "name: T-shirt\ndescription: A comfortable T-shirt\nprice: 100\nsunscreen: Yes\n\nname: Sunscreen\ndescription: A bottle of sunscreen\nprice: 500\nsunscreen: Yes\n\nname: Jeans\ndescription: A pair of jeans\nprice: 200\nsunscreen: No\n\nname: Melon\ndescription: A fresh melon\nprice: 1400\nsunscreen: No"
}
[llm/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: Use the following pieces of context to answer the user's question.\nIf you don't know the answer, just say that you don't know, don't try

'```markdown\n| 商品名称 | 描述           | 价格 | 防晒功能 |\n|----------|----------------|------|----------|\n| T-shirt  | A comfortable T-shirt | 100  | 是       |\n\n**总结：**  \n在提供的商品中，仅有一款T恤具备防晒功能，即“T-shirt”，价格为100元。\n```'

In [19]:
set_debug(False)

In [20]:
for i, e in enumerate(examples):
    print(i, type(e), e if not isinstance(e, dict) else e.keys())

0 <class 'dict'> dict_keys(['query', 'answer'])
1 <class 'dict'> dict_keys(['query', 'answer'])
2 <class 'dict'> dict_keys(['query', 'answer'])
3 <class 'dict'> dict_keys(['query', 'answer'])
4 <class 'dict'> dict_keys(['query', 'answer'])
5 <class 'dict'> dict_keys(['query', 'answer'])


In [21]:
predictions = qa.apply(examples)



> Entering new RetrievalQA chain...


C:\Users\35186\AppData\Local\Temp\ipykernel_31768\1205324748.py:1: LangChainDeprecationWarning: The method `Chain.apply` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `batch` instead.
  predictions = qa.apply(examples)



> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [22]:
from langchain_classic.evaluation.qa import QAEvalChain

In [23]:
eval_chain = QAEvalChain.from_llm(
    llm
)

In [24]:
graded_outputs = eval_chain.evaluate(
    examples,
    predictions
)

In [25]:
print(type(graded_outputs[0]))
print(graded_outputs[0])
print(graded_outputs[0].keys())

<class 'dict'>
{'results': 'INCORRECT\n\n**Explanation:**  \nThe true answer is simply "T-shirt", implying that only one such item exists and should be listed. While the student\'s answer does include "T-shirt" as having sun protection functionality, it adds additional unsupported details (e.g., description, price, table formatting, and a summary) that go beyond the scope of the true answer. Since the question asks to list *all* T-shirts with sun protection functionality in a specific format (a markdown table with a summary), but the true answer provides only the product name without any such formatting or extra data, the student’s response introduces fabricated or unverified information.\n\nGrading is based on **factual accuracy** relative to the true answer. The true answer does not support the table structure, pricing, description, or summary — therefore, the student’s answer is **INCORRECT** because it contains information not present in or implied by the true answer.'}
dict_keys([

In [26]:
for i, eg in enumerate(examples):
    print(f'Example {i}:')
    print('Query:', eg['query'])
    print('Real Answer:', eg['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print("Predicted Score:", graded_outputs[i]['results'])
    print("-"*100)

Example 0:
Query: 列出所有有防晒功能的T恤，用markdown格式输出，一张表格中，含有总结
Real Answer: T-shirt
Predicted Answer: 根据提供的信息，以下是具有防晒功能的 T 恤列表。

```markdown
| 名称   | 描述             | 价格 | 防晒功能 |
|--------|------------------|------|----------|
| T-shirt | 一件舒适的T恤     | 100  | 是       |

**总结：**  
在列出的商品中，仅有一款 T-shirt 具备防晒功能，价格为 100 元。
```
Predicted Score: INCORRECT

**Explanation:**  
The true answer is simply "T-shirt", implying that only one such item exists and should be listed. While the student's answer does include "T-shirt" as having sun protection functionality, it adds additional unsupported details (e.g., description, price, table formatting, and a summary) that go beyond the scope of the true answer. Since the question asks to list *all* T-shirts with sun protection functionality in a specific format (a markdown table with a summary), but the true answer provides only the product name without any such formatting or extra data, the student’s response introduces fabricated or unverified information.
